# Chapter 4 — API Design
### ML Engineer (Production) Interview Playbook

**Topics:** REST · JWT · OAuth · Status Codes · Pagination · Rate Limiting · Idempotency

> An API is the contract between your service and its consumers (mobile apps, other services, internal
> teams). Good design means consistency, security, and predictable behavior. In a financial environment
> like bunq, two things matter doubly: security (correct authentication and authorization) and
> idempotency (guaranteeing a payment operation doesn't run twice). This chapter goes deep on exactly
> these.
>
> **Interview tip:** The API designer's mindset: always think from the consumer's perspective. A good
> API is predictable, liberal in what it accepts, precise in what it returns, and its failures are
> clear and recoverable.

## 4.1 REST and Design Principles

REST is an architectural style, not a protocol. A few key principles: resources are identified by
URLs, HTTP verbs convey the meaning of the operation, and communication is stateless — the server keeps
no client session state between requests, and every request is self-contained.

Two important properties of HTTP verbs you should know the difference between: **"Safe"** means it
doesn't modify data; **"Idempotent"** means executing it multiple times has the same effect as once.

| Verb | Use | Safe | Idempotent |
|---|---|---|---|
| GET | Read a resource | Yes | Yes |
| POST | Create a new resource | No | No |
| PUT | Fully replace a resource | No | Yes |
| PATCH | Partial update | No | No* |
| DELETE | Delete a resource | No | Yes |

**Note:** Why is PUT idempotent but POST isn't? PUT replaces the entire resource with a specified
value; run it a hundred times and the final state is the same. But POST creates a new resource every
time; running it three times means three resources. This distinction is the foundation of the
idempotency discussion for payments.

### Resource naming conventions

- Use plural nouns for collections: `/users`, `/accounts/{id}/transactions`
- Don't put a verb in the URL; the verb is the HTTP action: `POST /users`, not `/createUser`
- Show hierarchy through nesting: `/users/42/accounts`

### Q4.1 — What's the difference between PUT and PATCH, and which is idempotent?

PUT replaces the entire resource: the body must be the complete representation of the resource, and
fields not sent are usually cleared/reset. PATCH changes only part of the resource. PUT is inherently
idempotent because repeating it gives the same final state. PATCH isn't necessarily idempotent — e.g.
if a PATCH is a relative operation like "decrease balance by 10," repeating it changes the result. If a
PATCH sets an absolute value, it effectively becomes idempotent.

## 4.2 Status Codes

The HTTP status code is the first thing a client reads to understand the result. Using them correctly
is a mark of a mature API designer.

| Family | Meaning | Key examples |
|---|---|---|
| 2xx | Success | `200 OK`, `201 Created`, `202 Accepted`, `204 No Content` |
| 3xx | Redirection | `301 Moved`, `304 Not Modified` |
| 4xx | Client error | `400`, `401`, `403`, `404`, `409`, `422`, `429` |
| 5xx | Server error | `500`, `502`, `503`, `504` |

### Frequently asked distinctions

- **401 vs. 403:** 401 means "I don't know who you are" (not authenticated); 403 means "I know who you
  are, but you're not allowed to do this."
- **400 vs. 422:** 400 means the request is malformed; 422 means the structure is correct but the
  content failed validation (e.g. an invalid email).
- **201 vs. 202:** 201 means the resource was created (usually with a `Location` header); 202 means the
  request was accepted and is being processed asynchronously.
- **409 Conflict:** conflict with the resource's current state (e.g. signing up with a duplicate email,
  or a version conflict in optimistic locking).

**Interview tip:** Don't confuse a server error (5xx) with a client error (4xx). If the user's input is
bad, return 4xx (the client must fix it, and retrying is pointless); if the problem is on your side,
return 5xx (the client can retry later). This distinction affects retry behavior and alerting.

### Q4.2 — What status code do you return for a payment operation processed asynchronously?

`202 Accepted`, since the operation has been accepted but isn't yet complete. Alongside it I typically
return an id or a status-tracking URL (e.g. `/payments/{id}/status`) so the client can poll for the
final result or be notified via a webhook. If the operation completes immediately and a resource is
created, `201 Created` with an appropriate `Location` header is more suitable.

## 4.3 JWT and Authentication

JWT (JSON Web Token) is a signed, stateless token for authentication. It has three parts separated by
dots: Header (algorithm), Payload (claims like user id and expiry), and Signature. The server verifies
the signature with a key and needs no stored session — which is exactly what makes it attractive for
distributed systems.

In [ ]:
import jwt, datetime

SECRET = "..."  # from a secret manager

def create_access_token(user_id: str) -> str:
    payload = {
        "sub": user_id,
        "iat": datetime.datetime.utcnow(),
        "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=15),
        "scope": "read write",
    }
    return jwt.encode(payload, SECRET, algorithm="HS256")

def verify(token: str) -> dict:
    # exp and the signature are checked automatically; error -> invalid token
    return jwt.decode(token, SECRET, algorithms=["HS256"])


**Note:** A common security mistake: a JWT is signed, not encrypted. The payload is just
base64url and anyone can read it. Never put sensitive data (passwords, card details) in it. The
signature only prevents tampering, not reading.

### Access Token and Refresh Token

Because a JWT is stateless, revoking it is hard — it stays valid until it expires. The standard
solution: a short-lived access token (e.g. 15 minutes) + a long-lived refresh token that's stored
server-side and revocable. When the access token expires, the client uses the refresh token to get a
new one.

### HS256 vs. RS256

- **HS256 (symmetric):** a single shared secret key for signing and verification. Simple, but any
  service that can verify can also forge a token.
- **RS256 (asymmetric):** a private key signs, a public key verifies. Better for microservice
  architectures because verifying services only hold the public key and can't forge tokens.

**Interview tip:** Storing the token on the client: `localStorage` is vulnerable to XSS attacks. An
`httpOnly` cookie is more resistant to XSS but requires CSRF protection. In a financial context,
`httpOnly` cookie + CSRF protection is the safer approach.

### Q4.3 — How do you revoke a JWT (logout/revoke) in a stateless system?

Since a JWT is inherently stateless, it stays valid until expiry and can't be directly revoked. A few
solutions: (1) Keep the access token's lifetime short (e.g. 5-15 minutes) and use a revocable refresh
token for renewal; on logout, revoke the refresh token server-side. (2) Keep a blacklist of revoked
tokens in Redis until expiry (this partly sacrifices statelessness). (3) Use a version identifier in
the claims that changes when the user changes their password, invalidating old tokens. In practice, a
short-lived access token + revocable refresh token is the most common combination.

## 4.4 OAuth 2.0 and OpenID Connect

OAuth 2.0 is an authorization framework, not authentication. It lets an application get limited access
to a user's resources without obtaining the user's password. It has four roles:

- **Resource Owner:** the user who owns the data.
- **Client:** the application that wants to access the data.
- **Authorization Server:** verifies identity and issues an access token.
- **Resource Server:** the protected API that accepts the token.

### Main flows (grant types)

- **Authorization Code + PKCE:** the safest for user-facing apps (SPAs, mobile); PKCE prevents
  interception of the code.
- **Client Credentials:** for service-to-service (machine-to-machine) communication with no user
  involved.

**Note:** The key distinction that's always asked: OAuth = authorization. OpenID Connect (OIDC) is a
layer on top of OAuth that adds authentication and provides an `id_token` (which is a JWT) so you know
"who the user is." If you see "Sign in with Google," that's OIDC, not plain OAuth.

### Q4.4 — What's the difference between OAuth and simple JWT-based authentication?

Simple JWT-based authentication means your own service verifies the user's identity and issues a
token — suitable when you own the users. OAuth is an authorization framework for when a third-party app
wants to access a user's resources (in another service) on their behalf, without obtaining the user's
password — instead of a password, it gets an access token with a limited scope. Note: OAuth itself
isn't authentication; for authentication you need OIDC (built on top of OAuth, providing an
`id_token`). JWT is simply a token format and can be used in either case.

## 4.5 Pagination

To avoid returning millions of rows in a single response, results are paginated. Two main methods with
different trade-offs:

| Method | Advantage | Drawback |
|---|---|---|
| Offset / Limit | Simple, jump to any page | Slow on deep pages; unstable if data changes between requests |
| Cursor / Keyset | Fast and stable even on deep pages | Can't jump directly to an arbitrary page |

In [ ]:
-- Offset (slow on deep pages because it reads and discards all prior rows)
SELECT * FROM transactions ORDER BY id LIMIT 20 OFFSET 100000;

-- Cursor / Keyset (fast and stable)
SELECT * FROM transactions
WHERE id > :last_seen_id
ORDER BY id
LIMIT 20;


**Interview tip:** Why is offset slow on deep pages? `OFFSET 100000` means the database must read
and discard 100,000 rows before reaching the ones you want. Keyset jumps straight to the right spot
using a condition on an indexed column. For feeds and large lists, keyset is the standard.

### Q4.5 — Why do you prefer cursor pagination over offset?

Two reasons: (1) Performance — offset must scan and discard all prior rows on deep pages (O(offset)),
while keyset jumps directly using a condition on an indexed column and stays constant. (2) Stability —
if a row is added/removed between page requests, offset can cause duplicated or skipped items; keyset,
being anchored to a value (like the last-seen id), stays stable. Its drawback is that jumping directly
to "page 50" isn't possible, which isn't a problem for infinite feeds.

## 4.6 Rate Limiting

Rate limiting prevents abuse, attacks, and overload (and cost). A few common algorithms with different
behavior:

| Algorithm | Idea |
|---|---|
| Fixed Window | Count within a fixed time window; simple but unfair at window boundaries (burst) |
| Sliding Window | A moving window; fairer, more computationally expensive |
| Token Bucket | A bucket refilled at a fixed rate; allows controlled bursts |
| Leaky Bucket | Output at a fixed rate; smooths out traffic |

In a distributed system, the counter must be shared and atomic; Redis is a common choice because it has
fast, atomic operations.

In [ ]:
# Simple token/fixed window with Redis
def allow(user_id: str, limit=100, window=60) -> bool:
    key = f"rate:{user_id}:{int(time.time() // window)}"
    count = redis.incr(key)
    if count == 1:
        redis.expire(key, window)
    return count <= limit


**Interview tip:** When the limit is exceeded, return `429 Too Many Requests` with a
`Retry-After` header (seconds until it resets). `X-RateLimit-Limit` and `X-RateLimit-Remaining` headers
also help the client adjust its behavior and avoid blindly retrying.

### Q4.6 — How do you implement rate limiting in a service with multiple instances?

The problem is that each instance has its own local counter, so a user could get a separate quota from
each instance. Solution: a single shared central counter, typically in Redis, that all instances hit.
The operation must be atomic (`INCR` is atomic, or for more complex algorithms, a Lua script that runs
several commands atomically). For better precision I use a sliding window log/counter. The limit can
also be layered: per-user, per-IP, and per-endpoint. You have to be careful that Redis itself doesn't
become a single point of failure.

## 4.7 Idempotency

An idempotent operation means running it multiple times has the same effect as running it once. This
is critical for payments: if a client resends a request due to a timeout or network drop, money must
not be deducted twice. The industry-standard solution: an `Idempotency-Key` that the client generates
and sends in a header.

In [ ]:
from fastapi import Header, HTTPException

@app.post("/payments")
async def pay(req: PaymentRequest, idempotency_key: str = Header(...)):
    # 1) If this key has already been processed, return the previous response
    existing = await store.get(idempotency_key)
    if existing is not None:
        return existing

    # 2) Process, and save the result under the same key (atomically)
    result = await process_payment(req)
    await store.save(idempotency_key, result)
    return result


**Note:** A concurrency challenge: if two requests with the same `Idempotency-Key` arrive
simultaneously, both might see `existing` as `None` and process twice. Solution: use a `UNIQUE`
constraint on the `idempotency_key` column in the database; the second insert is rejected with a
uniqueness error, telling you it's a duplicate. This is safer than a manual check.

### The relationship between HTTP verbs and idempotency

`GET`, `PUT`, and `DELETE` are idempotent by definition; `POST` isn't. This is exactly why sensitive
`POST`s (like creating a payment) need the `Idempotency-Key` mechanism to be safe against retries.

### Q4.7 — How do you guarantee a payment isn't executed twice due to a retry?

With the Idempotency-Key pattern: the client generates a unique key (e.g. a UUID) for each payment
operation and sends it in a header. Before processing, the server checks whether this key has already
been seen; if so, it returns the previous response without re-executing. To be safe against
concurrency, I store the key with a `UNIQUE` constraint in the database so two simultaneous requests
can't both be processed — the second insert fails with a uniqueness error. The entire operation
(recording the key + deducting money) must also be in a single atomic transaction to avoid an
inconsistent state.

## 4.8 Versioning and API Contract

A public API is a contract; breaking it breaks consumers. For evolving without breaking things,
versioning and backward compatibility are needed.

- **Version in the path:** `/v1/users`, `/v2/users` — common and explicit.
- **Version in a header:** via the `Accept` header or a custom one; cleaner but less discoverable.
- **Non-breaking change:** adding an optional field or a new endpoint — doesn't need a new version.
  Removing or changing a field's meaning is breaking.

**Interview tip:** Design a uniform error contract: all errors should have a consistent structure
(e.g. `{"error": code, "message": ..., "request_id": ...}`). This makes client-side error handling
predictable and avoids parsing variable text messages.

### Q4.8 — You want to remove a field from an API response. How do you do it without breaking consumers?

Removing a field is a breaking change, so it can't be done directly. Safe approach: (1) Mark the field
as deprecated and note it in the documentation, maybe with a warning header. (2) Keep both (the old
field + the new behavior) for a while so consumers can migrate. (3) If the change is large, release a
new API version (v2) and keep v1 alive for a transition period. Key point: breaking changes must come
with versioning, clear communication, and a migration window, not happen suddenly.

## Quick Chapter Summary (Cheat Sheet)

Review this on interview day:

- **REST:** Resource-oriented, stateless; `GET`/`PUT`/`DELETE` idempotent, `POST` isn't; `PUT` full
  replace, `PATCH` partial.
- **Status:** 401 (unknown identity) vs. 403 (no permission); 400 vs. 422; 201 vs. 202; 409 for
  conflict; 429 for rate.
- **JWT:** Signed, not encrypted; don't put sensitive data in it; short-lived access + revocable
  refresh; RS256 for microservices.
- **OAuth:** Authorization, not authentication; OIDC is the authentication layer with an `id_token`;
  Auth Code+PKCE for user apps, Client Credentials for m2m.
- **Pagination:** keyset/cursor is fast and stable; offset is slow and unstable on deep pages.
- **Rate limit:** Token/Sliding window; a central atomic counter in Redis; respond with 429 +
  `Retry-After`.
- **Idempotency:** `Idempotency-Key` for sensitive POSTs; a `UNIQUE` constraint for concurrency safety;
  within a single atomic transaction.
- **Versioning:** additions are safe, removal/meaning changes are breaking; version + a migration
  window; a uniform error structure.